# 7.22 — Self-supervised vision

Self-supervised vision turns an image into its own teacher: hide, crop, or perturb it, then make the representation keep the identity that should survive.

Self-supervised vision learns from images before class labels are supplied. Masked reconstruction, contrastive views, and teacher-student similarity all depend on embeddings that keep identity while ignoring harmless perturbations.

Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)

## The concept, built once: two views and InfoNCE

Patchifying a $4	imes4$ image into $2	imes2$ patches gives a $4	imes4$ table. Contrastive learning rewards a positive pair with cosine similarity near one and uses

$$-\log rac{\exp(s_+/	au)}{\exp(s_+/	au)+\exp(s_-/	au)}$$

to audit the batch.

In [ ]:
def patchify(image, patch_size):
    patches = []
    for r in range(0, image.shape[0], patch_size):
        for c in range(0, image.shape[1], patch_size):
            patches.append(image[r:r + patch_size, c:c + patch_size].ravel())
    return np.array(patches)

def cosine(a, b):
    return float(a.dot(b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def info_nce(pos, neg, tau):
    logits = np.array([pos, neg]) / tau
    logits = logits - logits.max()
    probs = np.exp(logits) / np.exp(logits).sum()
    return float(-np.log(probs[0]))

def self_supervised_embed(image):
    table = patchify(image, 2)
    positions = 0.03 * np.arange(table.shape[0])[:, None] * np.ones_like(table)
    masked = table + positions
    masked[1] = 0
    z1 = np.array([1.0, 2.0])
    z2 = np.array([1.144, 2.0])
    pos = cosine(z1, z2)
    loss = info_nce(pos, 0.20, 0.5)
    return table, positions, masked, pos, loss

image = np.arange(16).reshape(4, 4) / 15
table, positions, masked, pos, loss = self_supervised_embed(image)
assert table.shape == (4, 4)
assert np.allclose(positions[1] - positions[0], [0.03, 0.03, 0.03, 0.03])
assert round(pos, 4) == 0.9984
assert round(loss, 3) == 0.184
print("patch table", table.shape)
print("position gap", np.round(positions[1] - positions[0], 2))
print("positive cosine", round(pos, 4))
print("InfoNCE", round(loss, 3))

## Reusable embedding for a linear probe

A tiny self-supervised embedding concatenates safe augmented views: the original image, a one-pixel roll, and masked patch means. The label is used only by the linear probe evaluation.

In [ ]:
def ssl_embed(img):
    rolled = np.roll(img, shift=1, axis=1)
    patches = patchify(img, 2)
    patch_means = patches.mean(axis=1)
    patch_stds = patches.std(axis=1)
    masked_means = patch_means.copy()
    masked_means[::2] = 0
    global_stats = np.array([img.mean(), img.std(), cosine(img.ravel() + 1e-6, rolled.ravel() + 1e-6)])
    return np.concatenate([patch_means, patch_stds, masked_means, global_stats])

## Dataset ladder: D1 to D5

The shared `cv_ladder.py` ladder gives one CPU-safe interface: every rung returns images `X` with shape `(n, 8, 8)` and labels `y`. The same featurizer is used on every rung, so the accuracy curve measures scaling rather than a changing task.

In [ ]:
"""
F6 (Vision) shared dataset ladder — D1..D5 of rising complexity, CPU-only and run-all-safe.

This is the canonical ladder inlined into the classification-style Part-7 notebooks. Every
rung returns (X, y) with X shape (n, 8, 8) float in [0, 1] and integer labels y, so one
featurizer + classifier can run unchanged across all five rungs (the "watch it scale" story).

D4/D5 load real MNIST / CIFAR-10 via torchvision when the download is available (as in Colab) —
offline they fall back to a harder synthetic set so run-all never fails. Code is written one
statement per line for readability.
"""

import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


def _resize_to_8x8(img):
    """Nearest-neighbour resize of a 2-D array to 8x8 (no SciPy dependency)."""
    h, w = img.shape
    rows = (np.linspace(0, h - 1, 8)).round().astype(int)
    cols = (np.linspace(0, w - 1, 8)).round().astype(int)
    return img[np.ix_(rows, cols)]


def _normalize(x):
    """Scale an array into [0, 1] — a flat array becomes all zeros."""
    x = x.astype(float)
    lo = x.min()
    hi = x.max()
    if hi - lo < 1e-12:
        return np.zeros_like(x)
    return (x - lo) / (hi - lo)


def d1_hand_patches():
    """D1 — hand-built 4x4 patches, 2 classes: a vertical line (col 1) vs a horizontal line (row 1).

    Fixed positions with light jitter, so the two classes are cleanly separable and the
    mechanism is fully visible — the easy first rung.
    """
    rng = np.random.default_rng(0)
    images = []
    labels = []
    for _ in range(24):
        patch = rng.uniform(0.0, 0.15, size=(4, 4))
        patch[:, 1] = rng.uniform(0.85, 1.0)
        images.append(_resize_to_8x8(patch))
        labels.append(0)
    for _ in range(24):
        patch = rng.uniform(0.0, 0.15, size=(4, 4))
        patch[1, :] = rng.uniform(0.85, 1.0)
        images.append(_resize_to_8x8(patch))
        labels.append(1)
    return np.array(images), np.array(labels)


def d2_synthetic_shapes():
    """D2 — clean synthetic shapes on an 8x8 grid, 2 classes (square vs disc)."""
    rng = np.random.default_rng(1)
    yy, xx = np.mgrid[0:8, 0:8]
    images = []
    labels = []
    for _ in range(80):
        img = rng.uniform(0.0, 0.1, size=(8, 8))
        img[2:6, 2:6] = 0.9
        images.append(img)
        labels.append(0)
    for _ in range(80):
        img = rng.uniform(0.0, 0.1, size=(8, 8))
        disc = (xx - 3.5) ** 2 + (yy - 3.5) ** 2 <= 4.0
        img[disc] = 0.9
        images.append(img)
        labels.append(1)
    return np.array(images), np.array(labels)


def d3_sklearn_digits():
    """D3 — real sklearn digits (native 8x8), 4 classes for a fast, honest multi-class rung."""
    digits = load_digits()
    keep = np.isin(digits.target, [0, 1, 2, 3])
    X = digits.images[keep]
    y = digits.target[keep]
    X = np.array([_normalize(img) for img in X])
    return X, y


def _synthetic_textured(n_per_class, n_classes, noise, seed):
    """A harder synthetic fallback: textured class prototypes at 8x8 with noise."""
    rng = np.random.default_rng(seed)
    protos = [rng.uniform(0.0, 1.0, size=(8, 8)) for _ in range(n_classes)]
    images = []
    labels = []
    for cls in range(n_classes):
        for _ in range(n_per_class):
            img = protos[cls] + rng.normal(0.0, noise, size=(8, 8))
            images.append(_normalize(img))
            labels.append(cls)
    return np.array(images), np.array(labels)


def _call_with_timeout(fn, seconds):
    """Run fn() but abort with TimeoutError after `seconds` (guards slow/hanging downloads)."""
    import signal

    def _raise(signum, frame):
        raise TimeoutError("download timed out")

    old = signal.signal(signal.SIGALRM, _raise)
    signal.alarm(seconds)
    try:
        return fn()
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old)


def _load_mnist_gray(classes, n_per_class, seed, shift=False, noise=0.0):
    """Load MNIST via torchvision, grayscale + resize to 8x8, subsample. Raises on failure.

    MNIST is a small (~11 MB) real dataset. CIFAR-10 is deliberately avoided (a 170 MB
    download breaks run-all-safety) — the harder D5 rung instead shifts and noises MNIST.
    """
    import torchvision

    ds = torchvision.datasets.MNIST(root="./data", train=True, download=True)
    rng = np.random.default_rng(seed)
    targets = np.array(ds.targets)
    images = []
    labels = []
    for cls in classes:
        idx = np.where(targets == cls)[0][:n_per_class]
        for i in idx:
            arr = np.asarray(ds[int(i)][0], dtype=float)
            small = _resize_to_8x8(arr)
            if shift:
                small = np.roll(small, rng.integers(-1, 2), axis=0)
                small = np.roll(small, rng.integers(-1, 2), axis=1)
            if noise:
                small = small + rng.normal(0.0, noise * 255.0, size=(8, 8))
            images.append(_normalize(small))
            labels.append(cls)
    return np.array(images), np.array(labels)


def d4_mnist_or_fallback():
    """D4 — real MNIST (4 clean classes) when downloadable — else a harder synthetic set."""
    try:
        X, y = _call_with_timeout(lambda: _load_mnist_gray([0, 1, 2, 3], 60, seed=4), 30)
        return (X, y), "MNIST (real)"
    except Exception:
        return _synthetic_textured(60, 4, noise=0.35, seed=4), "synthetic (offline fallback)"


def d5_mnist_hard_or_fallback():
    """D5 — real MNIST, more classes with shift + noise (distribution shift) — else hardest synthetic."""
    try:
        X, y = _call_with_timeout(lambda: _load_mnist_gray([0, 1, 2, 3, 4, 5], 60, seed=5, shift=True, noise=0.12), 30)
        return (X, y), "MNIST shifted+noisy (real, harder)"
    except Exception:
        return _synthetic_textured(60, 6, noise=0.6, seed=5), "synthetic (offline fallback)"


def load_ladder():
    """Return the five rungs as a list of (name, X, y). D4/D5 note whether real data loaded."""
    rungs = []
    rungs.append(("D1 hand patches", *d1_hand_patches()))
    rungs.append(("D2 synthetic shapes", *d2_synthetic_shapes()))
    rungs.append(("D3 sklearn digits", *d3_sklearn_digits()))
    (x4, y4), tag4 = d4_mnist_or_fallback()
    rungs.append((f"D4 {tag4}", x4, y4))
    (x5, y5), tag5 = d5_mnist_hard_or_fallback()
    rungs.append((f"D5 {tag5}", x5, y5))
    return rungs


def accuracy_with(featurize, X, y):
    """Map each image through featurize, train logistic regression, return held-out accuracy."""
    feats = np.array([featurize(img) for img in X])
    x_tr, x_te, y_tr, y_te = train_test_split(feats, y, test_size=0.4, random_state=0, stratify=y)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf.score(x_te, y_te)


if __name__ == "__main__":
    for name, X, y in load_ladder():
        print(f"{name:38s} X={X.shape} classes={sorted(set(y.tolist()))} acc(flatten)={accuracy_with(lambda im: im.ravel(), X, y):.3f}")


In [ ]:
rungs = load_ladder()

fig, axes = plt.subplots(1, 5, figsize=(11, 2.4))
for ax, (name, X, y) in zip(axes, rungs):
    ax.imshow(X[0], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{name}\n{X.shape}, {len(set(y))} classes", fontsize=8)
    ax.axis("off")
plt.tight_layout()

for name, X, y in rungs:
    classes = sorted(set(y.tolist()))
    print(f"{name:34s} shape={X.shape} classes={classes}")

In [ ]:
ssl_scores = []
for name, X, y in rungs:
    score = accuracy_with(ssl_embed, X, y)
    ssl_scores.append(score)
    print(f"{name:34s} linear-probe accuracy={score:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(11, 2.5))
for ax, (name, X, y) in zip(axes, rungs):
    original = X[0]
    view = np.roll(original, shift=1, axis=1)
    panel = np.concatenate([original, view], axis=1)
    ax.imshow(panel, cmap="gray", vmin=0, vmax=1)
    ax.set_title(name.split()[0] + " two views", fontsize=8)
    ax.axis("off")
plt.tight_layout()

plt.figure(figsize=(5, 3))
plt.plot(range(1, 6), ssl_scores, marker="o")
plt.xticks(range(1, 6), ["D1", "D2", "D3", "D4", "D5"])
plt.ylim(0, 1.05)
plt.ylabel("linear-probe accuracy")
plt.title("Self-supervised embeddings across the ladder")
plt.grid(True, alpha=0.3)
plt.show()

## Pitfall on D5: unsafe positives

A crop that removes the object asks the embedding to match two views that no longer contain the same label-like evidence. High cosine alone is not semantic proof; compare against negatives and use safer augmentations.

In [ ]:
name, X, y = rungs[-1]
img = X[0]
bad_crop = np.zeros_like(img)
bad_crop[:4, :4] = img[:4, :4]
safe_view = np.roll(img, shift=1, axis=0)
bad_cos = cosine(ssl_embed(img), ssl_embed(bad_crop) + 1e-6)
safe_cos = cosine(ssl_embed(img), ssl_embed(safe_view) + 1e-6)
negative_cos = cosine(ssl_embed(img), ssl_embed(X[-1]) + 1e-6)
print("crop-removes-object cosine", round(float(bad_cos), 3))
print("safe augmentation cosine", round(float(safe_cos), 3))
print("negative-check cosine", round(float(negative_cos), 3))

## Evaluate it + Practice

- Metric: linear-probe accuracy; compare with the no-skill baseline, majority-class guessing.
- Cheap sanity check: D1 embedding should separate the two line classes.
- Ablation: remove positions or mask all patches.
- Failure signals: positive pairs disagree with the visible object.

Practice:
1. Change one D1 number and predict the metric before running it.

In [ ]:
# Your experiment here

2. Add one harder case to the ladder and rerun the curve.

In [ ]:
# Your harder case here

3. Turn off the key fix in the pitfall cell and explain the change.

In [ ]:
# Your ablation here